# Tahap 1 — Segmentasi Paru (HybridGNet Beku) & Uji Kualitas Mask

**File ini bertujuan untuk:**
1. Memuat model segmentasi paru pralatih **HybridGNet (CheXmask)** dengan seluruh bobot dibekukan.
2. Menguji kualitas *lung mask* terhadap anotasi dokter radiolog pada 85 citra RSUA (target DSC >= 0,85 dan IoU >= 0,75 per kelas).
3. Membuat *cache* satu kali citra 224 x 224 dan *lung mask* biner untuk seluruh 15.188 citra gabungan.
4. Mengevaluasi proporsi luas paru, keseimbangan paru kiri-kanan, sampel uji Kermany, dan mencatat sampel bermasalah.

**Daftar Isi:**
- [1. Persiapan Lingkungan & Cek Pembagian Data](#1-persiapan-lingkungan-cek-pembagian-data)
- [2. Tahap 1 — HybridGNet Beku (Pemuatan Bobot)](#2-tahap-1-hybridgnet-beku-pemuatan-bobot)
- [3. Tahap 1 — Uji Kualitas Mask](#3-tahap-1-uji-kualitas-mask)
  - [3.1 Evaluasi Kualitas pada Dataset RSUA (85 Citra)](#31-evaluasi-kualitas-pada-dataset-rsua-85-citra)
  - [3.2 Pembuatan Cache Citra 224x224 & Lung Mask](#32-pembuatan-cache-citra-224x224-lung-mask)
  - [3.3 Evaluasi Luas Paru, Keseimbangan Kiri-Kanan, & Sampel Uji Kermany](#33-evaluasi-luas-paru-keseimbangan-kiri-kanan-sampel-uji-kermany)
  - [3.4 Daftar Sampel di Bawah Batas Kualitas](#34-daftar-sampel-di-bawah-batas-kualitas)
- [4. Ringkasan Pemeriksaan Akhir (Gate Check)](#4-ringkasan-pemeriksaan-akhir-gate-check)

<a id="1-persiapan-lingkungan-cek-pembagian-data"></a>
## 1. Persiapan Lingkungan & Cek Pembagian Data

Menyiapkan lingkungan eksekusi, memuat pengaturan proyek, dan memverifikasi integritas berkas `split_patientwise.csv`.
- `verify_locked_split()`: memastikan file pembagian data belum berubah lewat *checksum* SHA-256.
- `set_global_seed()`: mengunci angka acak agar pemilihan sampel evaluasi selalu sama.

In [ ]:
# Sel 1: Setup lingkungan seragam (Colab/Lokal), config terpusat, seed deterministik, & cek split terkunci
import json
import logging
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive  # type: ignore
    if not Path("/content/drive/MyDrive").exists():
        drive.mount("/content/drive")
    # Pasang hanya pydicom (HybridGNet pure-PyTorch tidak perlu smp atau torch_geometric)
    !pip install -q pydicom

for candidate in [
    Path.cwd(),
    Path.cwd().parent,
    Path("/content/drive/MyDrive/LAB_TA_GradCAM_Pneumonia/code"),
    Path("/content/gradcam-pneumonia"),
]:
    if (candidate / "src" / "paths.py").exists():
        if str(candidate) not in sys.path:
            sys.path.insert(0, str(candidate))
        break

import pandas as pd
import torch
from IPython.display import display

from src.data.split import verify_locked_split
from src.paths import ensure_output_dirs, load_config
from src.seed import get_env_snapshot, print_env_banner, set_global_seed

logging.basicConfig(level=logging.INFO, format="[%(levelname)s] %(message)s")

CFG = load_config()
PATHS = ensure_output_dirs(cfg=CFG)
PROJECT_ROOT = PATHS["project_root"]
FIG_QC_DIR = PATHS["outputs"] / "figures" / "qc"
TAB_QC_DIR = PATHS["outputs"] / "tables" / "qc"

# Konfigurasi terpusat dari configs/config.yaml (tanpa magic number)
LM_LOCKED = CFG["locked"]["lung_mask"]
PP_LOCKED = CFG["locked"]["preprocess"]
QC_CFG = {
    "seed": int(CFG["assumed"]["qc_kermany_selection_seed"]["value"]),
    "image_size": int(PP_LOCKED["image_size"]["value"]),
    "binarize_thr": float(LM_LOCKED["binarize_threshold"]["value"]),
    "keep_components": int(LM_LOCKED["keep_largest_components"]["value"]),
    "fill_holes": bool(LM_LOCKED["hole_filling"]["value"]),
    "dsc_min": float(LM_LOCKED["qc_dsc_min"]["value"]),
    "iou_min": float(LM_LOCKED["qc_iou_min"]["value"]),
    "rsua_normal_n": int(LM_LOCKED["qc_rsua_normal"]["value"]),
    "rsua_pneumonia_n": int(LM_LOCKED["qc_rsua_pneumonia"]["value"]),
    "rsua_covid_excluded": int(LM_LOCKED["rsua_covid_excluded"]["value"]),
    "kermany_qc_per_class": int(LM_LOCKED["qc_kermany_normal"]["value"]),
    "r_paru_extreme_thr": float(LM_LOCKED["r_paru_extreme_threshold"]["value"]),
    "r_paru_diff_thr": float(LM_LOCKED["r_paru_class_diff_threshold"]["value"]),
    "lr_balance_min_ratio": float(LM_LOCKED.get("lr_balance_min_ratio", {}).get("value", 0.30)),
    "debug_mode": bool(CFG["runtime"]["DEBUG"]),
    "debug_max_cache": int(CFG["runtime"]["debug_subset_per_class"]) * 2,
}

set_global_seed(QC_CFG["seed"], deterministic=True)
ENV_SNAPSHOT = get_env_snapshot(seed=QC_CFG["seed"])
print_env_banner(ENV_SNAPSHOT)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SPLIT_CSV = PATHS["interim"] / "split_patientwise.csv"
verify_locked_split(SPLIT_CSV)
SPLIT_DF = pd.read_csv(SPLIT_CSV)

# Deteksi cepat direktori akar citra (ekstraksi otomatis RSNA/PNG hanya dijalankan di Sel 4 bila dibutuhkan)
first_rel = str(SPLIT_DF["rel_path"].iloc[0])
work_harm = Path("/content/work/harmonized_png")
if IN_COLAB and (work_harm / first_rel).exists():
    IMAGE_ROOT = work_harm
elif (PATHS["harmonized_png"] / first_rel).exists():
    IMAGE_ROOT = PATHS["harmonized_png"]
else:
    IMAGE_ROOT = work_harm if IN_COLAB else PATHS["kermany_chest_xray"]

env_path = PATHS["outputs"] / "logs" / "env_snapshot_nb02.json"
env_path.write_text(json.dumps(ENV_SNAPSHOT, indent=2), encoding="utf-8")

print(f"\nIN_COLAB         : {IN_COLAB}")
print(f"PROJECT_ROOT     : {PROJECT_ROOT}")
print(f"IMAGE_ROOT       : {IMAGE_ROOT}")
print(f"Split Terkunci   : {SPLIT_CSV} (n={len(SPLIT_DF)} citra, SHA-256 TERVERIFIKASI)")
print(f"DEBUG Mode       : {QC_CFG['debug_mode']}")

<a id="2-tahap-1-hybridgnet-beku-pemuatan-bobot"></a>
## 2. Tahap 1 — HybridGNet Beku (Pemuatan Bobot)

Memuat bobot pralatih publik HybridGNet (CheXmask) lalu mengunci 100% parameternya agar tidak berubah selama penelitian.
- `download_hybridgnet_weights()`: menyiapkan berkas bobot `weights.pt` di folder model.
- `load_frozen_hybridgnet()`: memuat model dalam mode evaluasi dengan jumlah parameter terlatih tepat 0.
- `get_hybridgnet_spec()`: mencatat detail model dan *checksum* bobot ke `seg_model_spec.csv`.

In [ ]:
# Sel 2: Unduh & muat bobot HybridGNet beku (CheXmask — Gaggion dkk., 2024) + dokumentasi format masukan
from src.segmentation.hybridgnet import (
    compute_file_sha256,
    download_hybridgnet_weights,
    get_hybridgnet_spec,
    load_frozen_hybridgnet,
)

weights_dir = PATHS["hybridgnet_frozen"]
weights_path = download_hybridgnet_weights(target_dir=weights_dir, force_download=False)
weights_sha256 = compute_file_sha256(weights_path)

unet_spec = get_hybridgnet_spec()
unet_spec["weights_path"] = str(weights_path)
unet_spec["weights_sha256"] = weights_sha256
unet_spec["weights_size_bytes"] = int(weights_path.stat().st_size)

spec_rows = [
    {"parameter": k, "nilai": json.dumps(v) if isinstance(v, (dict, list)) else str(v)}
    for k, v in unet_spec.items()
]
unet_spec_df = pd.DataFrame(spec_rows)
unet_spec_csv = TAB_QC_DIR / "seg_model_spec.csv"
unet_spec_df.to_csv(unet_spec_csv, index=False, encoding="utf-8")

print("=== DOKUMENTASI FORMAT MASUKAN & SUMBER MODEL HYBRIDGNET (CHEXMASK) ===")
display(unet_spec_df)
print(f"Tabel spesifikasi tersimpan di: {unet_spec_csv}\n")

unet_model = load_frozen_hybridgnet(weights_path, device=DEVICE)
total_params = sum(p.numel() for p in unet_model.parameters())
trainable_params = sum(p.numel() for p in unet_model.parameters() if p.requires_grad)

print(f"Total Parameter HybridGNet : {total_params:,} (~{total_params / 1e6:.2f} Juta)")
print(f"Parameter Dapat Dilatih    : {trainable_params} (WAJIB 0)")
assert trainable_params == 0, "[FATAL] Terdeteksi bobot HybridGNet yang tidak beku!"
print("[OK] Model HybridGNet (CheXmask) 100% BEKU (frozen) sesuai Bab III §3.2.2.")

<a id="3-tahap-1-uji-kualitas-mask"></a>
## 3. Tahap 1 — Uji Kualitas Mask

<a id="31-evaluasi-kualitas-pada-dataset-rsua-85-citra"></a>
### 3.1 Evaluasi Kualitas pada Dataset RSUA (85 Citra)

Membandingkan batas paru keluaran HybridGNet terhadap anotasi dokter radiolog pada 32 citra Normal dan 53 citra Pneumonia RSUA.
- `evaluate_rsua_dataset()`: menghitung nilai DSC dan IoU tiap citra serta status lulus per kelas.
- `plot_rsua_metrics_distribution()`: menampilkan sebaran nilai DSC dan IoU terhadap garis target.
- `plot_rsua_overlay_grid()`: menampilkan perbandingan visual garis paru dokter (cyan) dan model (hijau).

In [ ]:
# Sel 3: QC Kuantitatif & Visual pada Dataset RSUA Validated (85 citra: 32 Normal, 53 Pneumonia)
from src.segmentation.qc import evaluate_rsua_dataset
from src.viz.qc_plots import plot_rsua_metrics_distribution, plot_rsua_overlay_grid

rsua_root = PATHS["rsua_validated"]
print(f"Mengevaluasi batas paru U-Net pada dataset RSUA: {rsua_root}")

rsua_qc = evaluate_rsua_dataset(
    unet_model=unet_model,
    rsua_validated_root=rsua_root,
    device=DEVICE,
    dsc_min=QC_CFG["dsc_min"],
    iou_min=QC_CFG["iou_min"],
    binarize_thr=QC_CFG["binarize_thr"],
    keep_components=QC_CFG["keep_components"],
    fill_holes=QC_CFG["fill_holes"],
    expected_normal=QC_CFG["rsua_normal_n"],
    expected_pneumonia=QC_CFG["rsua_pneumonia_n"],
    r_paru_extreme_thr=QC_CFG["r_paru_extreme_thr"],
)

rsua_summary_df = rsua_qc["summary_df"]
rsua_details_df = rsua_qc["details_df"]

rsua_summary_csv = TAB_QC_DIR / "rsua_qc_summary.csv"
rsua_details_csv = TAB_QC_DIR / "rsua_qc_details.csv"
rsua_summary_df.to_csv(rsua_summary_csv, index=False, encoding="utf-8")
rsua_details_df.to_csv(rsua_details_csv, index=False, encoding="utf-8")
rsua_details_df.to_csv(PATHS["interim"] / "rsua_qc_details.csv", index=False, encoding="utf-8")

print(f"\n=== RINGKASAN QC RSUA (Target Proposal: DSC >= {QC_CFG['dsc_min']}, IoU >= {QC_CFG['iou_min']} per kelas) ===")
print(f"Status Keseluruhan RSUA QC : {rsua_qc['status']}")
display(rsua_summary_df)
print(f"Tabel ringkasan tersimpan : {rsua_summary_csv}")
print(f"Tabel detail tersimpan    : {rsua_details_csv}")

fig_rsua_dist = FIG_QC_DIR / "rsua_dsc_iou_distribution.png"
plot_rsua_metrics_distribution(
    rsua_details_df,
    dsc_min=QC_CFG["dsc_min"],
    iou_min=QC_CFG["iou_min"],
    save_path=fig_rsua_dist,
    show=True,
)
print(f"Figur distribusi tersimpan: {fig_rsua_dist}")

fig_rsua_ovl = FIG_QC_DIR / "rsua_overlay_random_vs_worst.png"
plot_rsua_overlay_grid(
    rsua_details_df=rsua_details_df,
    unet_model=unet_model,
    device=DEVICE,
    n_random=4,
    n_worst=4,
    seed=QC_CFG["seed"],
    binarize_thr=QC_CFG["binarize_thr"],
    keep_components=QC_CFG["keep_components"],
    fill_holes=QC_CFG["fill_holes"],
    save_path=fig_rsua_ovl,
    show=True,
)
print(f"Figur overlay RSUA tersimpan: {fig_rsua_ovl}")

<a id="32-pembuatan-cache-citra-224x224-lung-mask"></a>
### 3.2 Pembuatan Cache Citra 224x224 & Lung Mask

Menjalankan segmentasi paru satu kali untuk seluruh 15.188 citra (latih, validasi, uji) dan menyimpan hasilnya ke `data/processed/` agar pelatihan berjalan cepat.
- `ensure_harmonized_images_for_split()`: menyiapkan gambar split di penyimpanan lokal Colab bila sesi baru.
- `run_full_lung_mask_cache()`: menghasilkan dan menyimpan pasangan citra `I_224` dan *mask* biner `M_224` secara *resumable*.
- `verify_cache_integrity()`: mengecek kelengkapan berkas, dimensi 224 x 224, dan nilai biner *mask*.

In [ ]:
# Sel 4: Caching Satu-Kali I_224 (float32) & M_224 (uint8 biner) untuk Seluruh 15.188 Citra (docs/2.md Aturan 7)
# (Hanya jalankan Sel 4-7 setelah Bab III disetujui dosen agar hemat kuota GPU & ruang Drive)
from src.segmentation.cache import (
    PreprocessCacheConfig,
    run_full_lung_mask_cache,
    verify_cache_integrity,
)

# Siapkan PNG split di SSD lokal Colab secara otomatis bila runtime GPU baru
if IN_COLAB and not (IMAGE_ROOT / first_rel).exists() and "source" in SPLIT_DF.columns:
    from src.data.manifest import ensure_harmonized_images_for_split
    work_rsna = Path("/content/work/rsna")
    rsna_zip = PATHS["rsna_raw"] / "rsna-pneumonia-detection-challenge.zip"
    if not (work_rsna / "stage_2_train_images").exists() and rsna_zip.exists():
        import zipfile
        work_rsna.mkdir(parents=True, exist_ok=True)
        needed_pids = set(SPLIT_DF[SPLIT_DF["source"].astype(str).str.lower() == "rsna"]["patient_id"].astype(str))
        with zipfile.ZipFile(rsna_zip, "r") as zf:
            members = [
                m for m in zf.namelist()
                if m.startswith("stage_2_train_images/") and Path(m).stem in needed_pids
            ]
            print(f"Mengekstrak otomatis {len(members)} DICOM RSNA terpilih ke {work_rsna} ...")
            zf.extractall(work_rsna, members=members)
    rsna_src = work_rsna if (work_rsna / "stage_2_train_images").exists() else PATHS["rsna_raw"]
    IMAGE_ROOT = ensure_harmonized_images_for_split(
        split_df=SPLIT_DF,
        kermany_root=PATHS["kermany_chest_xray"],
        rsna_root=rsna_src,
        harmonized_dir=Path("/content/work/harmonized_png"),
        num_workers=8,
        show_progress=True,
    )

cache_cfg = PreprocessCacheConfig.from_yaml_cfg(CFG)
max_cache = QC_CFG["debug_max_cache"] if QC_CFG["debug_mode"] else None

print(f"Menjalankan caching resumable ke: {PATHS['processed']}")
if max_cache is not None:
    print(f"[MODE DEBUG AKTIF] Membatasi caching ke {max_cache} citra pertama.")

kermany_meta_df = run_full_lung_mask_cache(
    unet_model=unet_model,
    split_csv_path=SPLIT_CSV,
    kermany_root=IMAGE_ROOT,
    processed_dir=PATHS["processed"],
    cache_cfg=cache_cfg,
    device=DEVICE,
    verify_split_checksum=True,
    max_samples=max_cache,
    show_progress=True,
)

cache_check = verify_cache_integrity(
    split_csv_path=SPLIT_CSV,
    processed_dir=PATHS["processed"],
    sample_check_n=25,
    seed=QC_CFG["seed"],
    expected_size=QC_CFG["image_size"],
)
cache_check_df = pd.DataFrame([cache_check])
cache_check_csv = TAB_QC_DIR / "cache_integrity_check.csv"
cache_check_df.to_csv(cache_check_csv, index=False, encoding="utf-8")

print("\n=== VERIFIKASI INTEGRITAS CACHE data/processed/ ===")
display(cache_check_df)
print(f"Metadata penuh tersimpan di : {PATHS['processed'] / 'lung_mask_metadata.csv'}")

<a id="33-evaluasi-luas-paru-keseimbangan-kiri-kanan-sampel-uji-kermany"></a>
### 3.3 Evaluasi Luas Paru, Keseimbangan Kiri-Kanan, & Sampel Uji Kermany

Memeriksa sebaran proporsi luas paru (`r_paru`) serta keseimbangan paru kiri-kanan (`r_LR`) di tiap sumber dan kelas, sekaligus mengekspor 50 gambar overlay uji Kermany untuk verifikasi kontur manual.
- `summarize_full_r_paru()`: meringkas luas paru, kasus ekstrem, ketimpangan kiri-kanan, dan selisih antar-kelas.
- `select_kermany_qc_subset()` & `run_overlay_export()`: memilih 50 sampel uji Kermany (25 Normal, 25 Pneumonia) dan menyimpan gambar garis parunya.
- `evaluate_kermany_manual_masks()`: menghitung DSC dan IoU terhadap kontur manual penulis bila berkas sudah tersedia.

In [ ]:
# Sel 5: Analisis Penuh r_paru (L5 & docs/3.md butir 8), Ekspor Overlay 50 Sampel D_test, & QC Kontur Manual (L3(2))
from scripts.export_overlay_for_manual_masks import run_overlay_export
from src.segmentation.qc import (
    evaluate_kermany_manual_masks,
    select_kermany_qc_subset,
    summarize_full_r_paru,
)
from src.viz.qc_plots import plot_kermany_overlay_grid, plot_r_paru_histogram

r_paru_summary = summarize_full_r_paru(
    kermany_meta_df,
    r_paru_extreme_thr=QC_CFG["r_paru_extreme_thr"],
    r_paru_diff_thr=QC_CFG["r_paru_diff_thr"],
    lr_balance_min_ratio=QC_CFG["lr_balance_min_ratio"],
)
r_paru_split_csv = TAB_QC_DIR / "r_paru_by_split_class.csv"
r_paru_source_csv = TAB_QC_DIR / "r_paru_by_source_class.csv"
r_paru_overall_csv = TAB_QC_DIR / "r_paru_overall_by_class.csv"
r_paru_summary["per_split_class_df"].to_csv(r_paru_split_csv, index=False, encoding="utf-8")
r_paru_summary["per_source_class_df"].to_csv(r_paru_source_csv, index=False, encoding="utf-8")
r_paru_summary["overall_by_class_df"].to_csv(r_paru_overall_csv, index=False, encoding="utf-8")

print("=== RINGKASAN DISTRIBUSI r_paru & KESEIMBANGAN PARU KIRI-KANAN (PER SUMBER x KELAS) ===")
display(r_paru_summary["per_source_class_df"])
print("=== RINGKASAN DISTRIBUSI r_paru (PER SPLIT & KELAS) ===")
display(r_paru_summary["per_split_class_df"])
print(
    f"Seluruh data : |delta r_paru| = {r_paru_summary['delta_r_paru_all']:.4f} "
    f"(Ambang {QC_CFG['r_paru_diff_thr']}) -> Wajib ACR_norm: {r_paru_summary['triggers_acr_norm_all']} | "
    f"Ketimpangan kiri-kanan (r_LR<{QC_CFG['lr_balance_min_ratio']}): {r_paru_summary['total_lr_imbalances']}"
)
print(
    f"Khusus D_test: |delta r_paru| = {r_paru_summary['delta_r_paru_test']:.4f} "
    f"-> Wajib ACR_norm: {r_paru_summary['triggers_acr_norm_test']} | "
    f"Kegagalan ekstrem D_test (<{QC_CFG['r_paru_extreme_thr']}): {r_paru_summary['test_extreme_failures']}"
)

fig_rparu_hist = FIG_QC_DIR / "kermany_r_paru_histogram.png"
plot_r_paru_histogram(
    kermany_meta_df,
    r_paru_extreme_thr=QC_CFG["r_paru_extreme_thr"],
    r_paru_diff_thr=QC_CFG["r_paru_diff_thr"],
    save_path=fig_rparu_hist,
    show=True,
)
print(f"Figur histogram r_paru tersimpan: {fig_rparu_hist}")

fig_kerm_ovl = FIG_QC_DIR / "kermany_overlay_random_vs_worst.png"
plot_kermany_overlay_grid(
    metadata_df=kermany_meta_df,
    processed_dir=PATHS["processed"],
    n_random=4,
    n_worst=4,
    seed=QC_CFG["seed"],
    save_path=fig_kerm_ovl,
    show=True,
)
print(f"Figur overlay Kermany tersimpan : {fig_kerm_ovl}")

subset_50_csv = PATHS["interim"] / "kermany_qc_50_list.csv"
subset_50_df = select_kermany_qc_subset(
    SPLIT_DF,
    n_per_class=QC_CFG["kermany_qc_per_class"],
    seed=QC_CFG["seed"],
    out_csv=subset_50_csv,
)
overlay_dir = PATHS["outputs"] / "figures" / "lung_mask_overlays"
exported_overlays = run_overlay_export(
    n_samples_per_class=QC_CFG["kermany_qc_per_class"],
    output_dir=overlay_dir,
    unet_model=unet_model,
    device=DEVICE,
    seed=QC_CFG["seed"],
    image_root=IMAGE_ROOT,
)

manual_qc = evaluate_kermany_manual_masks(
    unet_model=unet_model,
    subset_df=subset_50_df,
    kermany_root=IMAGE_ROOT,
    manual_masks_dir=PATHS["kermany_manual_masks"],
    device=DEVICE,
    dsc_min=QC_CFG["dsc_min"],
    iou_min=QC_CFG["iou_min"],
    binarize_thr=QC_CFG["binarize_thr"],
    keep_components=QC_CFG["keep_components"],
    fill_holes=QC_CFG["fill_holes"],
)
print(f"\n=== STATUS QC KONTUR MANUAL KERMANY (50 SAMPEL D_test) ===")
print(f"Status : {manual_qc['status']} ({manual_qc['n_found']}/{manual_qc['n_expected']} berkas ditemukan)")
if not manual_qc["summary_df"].empty:
    display(manual_qc["summary_df"])
    manual_qc["summary_df"].to_csv(TAB_QC_DIR / "kermany_manual_qc_summary.csv", index=False, encoding="utf-8")
    manual_qc["details_df"].to_csv(TAB_QC_DIR / "kermany_manual_qc_details.csv", index=False, encoding="utf-8")
else:
    print(f"[INFO] {manual_qc.get('note', '')}")

<a id="34-daftar-sampel-di-bawah-batas-kualitas"></a>
### 3.4 Daftar Sampel di Bawah Batas Kualitas

Mencatat seluruh citra yang nilainya berada di bawah batas kualitas atau mengalami kegagalan deteksi ekstrem tanpa menghapusnya dari data uji.
- `build_failed_qc_table()`: mengumpulkan daftar sampel bermasalah beserta alasannya ke `qc_failed_samples.csv`.

In [ ]:
# Sel 6: Daftar Sampel yang Gagal Kriteria QC (RSUA & Kermany) Beserta Alasan
from src.segmentation.qc import build_failed_qc_table

failed_qc_df = build_failed_qc_table(
    rsua_details_df=rsua_details_df,
    kermany_meta_df=kermany_meta_df,
    dsc_min=QC_CFG["dsc_min"],
    iou_min=QC_CFG["iou_min"],
    r_paru_extreme_thr=QC_CFG["r_paru_extreme_thr"],
)
failed_qc_csv = TAB_QC_DIR / "qc_failed_samples.csv"
failed_qc_df.to_csv(failed_qc_csv, index=False, encoding="utf-8")

print(f"=== DAFTAR SAMPEL GAGAL QC (Total: {len(failed_qc_df)} sampel) ===")
if failed_qc_df.empty:
    print("[OK] Tidak ada sampel RSUA (DSC<0.85/IoU<0.75) maupun kegagalan ekstrem Kermany (r_paru<0.05).")
else:
    display(failed_qc_df.head(30))
    if len(failed_qc_df) > 30:
        print(f"... menampilkan 30 dari {len(failed_qc_df)} baris. Lihat berkas lengkap di: {failed_qc_csv}")
print(f"Daftar lengkap tersimpan di: {failed_qc_csv}")

<a id="4-ringkasan-pemeriksaan-akhir-gate-check"></a>
## 4. Ringkasan Pemeriksaan Akhir (Gate Check)

Menampilkan rangkuman status lulus/gagal (`PASS`/`FAIL`) seluruh syarat Tahap 1 sebelum lanjut ke tahap berikutnya.
- `json.dump()` & `display()`: menampilkan tabel ringkasan pemeriksaan dan menyimpan laporan ke `lung_mask_qc_report.json`.

In [ ]:
# Sel 7: GATE CHECK RESMI — Keputusan PASS/FAIL Notebook 02 (Tahap 1 Gambar 3.1 / Gerbang 3)
gate_rows = [
    {
        "kriteria": "1. Bobot HybridGNet (CheXmask) termuat & 100% beku (L1)",
        "acuan_docs": "Metodologi.tex L68; docs/1.md L1",
        "hasil": f"params={total_params:,}, trainable={trainable_params}",
        "status": "PASS" if trainable_params == 0 else "FAIL",
    },
    {
        "kriteria": "2. QC RSUA 85 citra (DSC>=0.85 & IoU>=0.75 per kelas) (L3(1), L4)",
        "acuan_docs": "Metodologi.tex L14, L71, L73; docs/1.md L4",
        "hasil": (
            f"Normal(DSC={rsua_qc['summary_by_class']['Normal']['dice_mean']:.4f}, "
            f"IoU={rsua_qc['summary_by_class']['Normal']['iou_mean']:.4f}) | "
            f"Pneumonia(DSC={rsua_qc['summary_by_class']['Pneumonia']['dice_mean']:.4f}, "
            f"IoU={rsua_qc['summary_by_class']['Pneumonia']['iou_mean']:.4f})"
        ),
        "status": rsua_qc["status"],
    },
    {
        "kriteria": "3. Ekspor overlay >=10 (target 50) sampel D_test Kermany (G3(2))",
        "acuan_docs": "docs/3.md Gerbang 3; Metodologi.tex L73",
        "hasil": f"{len(exported_overlays)} citra overlay tersimpan di {overlay_dir.name}/",
        "status": "PASS" if len(exported_overlays) >= 10 else "FAIL",
    },
    {
        "kriteria": "4. QC Kontur Manual 50 citra D_test Kermany (L3(2))",
        "acuan_docs": "Metodologi.tex L73; docs/1.md L3(2)",
        "hasil": f"{manual_qc['n_found']}/{manual_qc['n_expected']} berkas manual dievaluasi ({manual_qc['status']})",
        "status": manual_qc["status"],
    },
    {
        "kriteria": "5. Cache penuh I_224 (float32) & M_224 (uint8 biner) di data/processed/",
        "acuan_docs": "docs/2.md Aturan 7; AMBIG-17",
        "hasil": f"{cache_check['n_cached']}/{cache_check['n_expected']} citra, missing_files={cache_check.get('n_missing_files', -1)}",
        "status": cache_check["status"],
    },
    {
        "kriteria": "6. Evaluasi r_paru penuh, kegagalan ekstrem (<0.05), & cek ACR_norm (L5)",
        "acuan_docs": "Metodologi.tex L73, L77; docs/1.md L5",
        "hasil": f"|delta_r_test|={r_paru_summary['delta_r_paru_test']:.4f}, extreme_all={r_paru_summary['total_extreme_failures']}, wajib_ACR_norm={r_paru_summary['triggers_acr_norm_test']}",
        "status": "PASS",
    },
]

gate_df = pd.DataFrame(gate_rows)
gate_csv = TAB_QC_DIR / "gate_check_nb02.csv"
gate_df.to_csv(gate_csv, index=False, encoding="utf-8")

qc_report = {
    "gate": "GERBANG_3_LUNG_MASK_DAN_QC",
    "unet_spec": unet_spec,
    "rsua_qc": {"status": rsua_qc["status"], "summary": rsua_qc["summary_by_class"]},
    "kermany_manual_qc": {
        "status": manual_qc["status"],
        "n_found": manual_qc["n_found"],
        "n_expected": manual_qc["n_expected"],
        "summary": manual_qc["summary_by_class"],
    },
    "kermany_r_paru": {k: v for k, v in r_paru_summary.items() if not isinstance(v, pd.DataFrame)},
    "cache_integrity": cache_check,
    "n_failed_qc_samples": int(len(failed_qc_df)),
    "gate_check": gate_rows,
}
qc_report_path = PATHS["interim"] / "lung_mask_qc_report.json"
qc_report_path.write_text(json.dumps(qc_report, indent=2), encoding="utf-8")

print("===================================================================")
print("                  GATE CHECK — NOTEBOOK 02 (LUNG MASK & QC)        ")
print("===================================================================")
display(gate_df)

if rsua_qc["status"] == "FAIL" or manual_qc["status"] == "FAIL":
    print("\n" + "!" * 72)
    print("PERINGATAN BESAR (docs/1.md L4 — Metodologi.tex L73):")
    print("QC segmentasi paru TIDAK MEMENUHI ambang DSC >= 0.85 / IoU >= 0.75!")
    print("Sesuai batasan proposal: JANGAN melatih ulang U-Net.")
    print("Untuk tetap melanjutkan ke notebook 04_train_all, WAJIB set")
    print("`ACKNOWLEDGE_QC_FAIL: true` secara sadar di `configs/config.yaml`.")
    print("!" * 72)

print("\n=== RINGKASAN OUTPUT NOTEBOOK 02 ===")
print(f"Input  : {SPLIT_CSV}, {PATHS['rsua_validated']}")
print(f"Output : {qc_report_path}")
print(f"Cache  : {PATHS['processed']} (images_224/, masks_224/, lung_mask_metadata.csv)")
print(f"Tabel  : {TAB_QC_DIR}")
print(f"Figur  : {FIG_QC_DIR} & {overlay_dir}")